<a href="https://colab.research.google.com/github/ssmartin-code/MVMC/blob/main/1_ConsultaMasiva.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Consulta Masiva

Para poder obtener un listado de referencias catatsrales hace falta guardar el shapefile de las parcelas en particular el archivo .dbf en la carpeta 0.Descarga datos del Intersect. El nombre del shape al guardar tiene que ser RF_{Monte}



In [ ]:
# Variables
monte = "Bullaso"

In [ ]:
!pip install geopandas fiona shapely pyproj
import geopandas as gpd


In [ ]:
# Ruta al archivo DBF
ruta_archivo = f"/content/drive/MyDrive/Catastro/{monte}/0.Descarga datos del Intersect/Vectorial_{monte}.dbf"

# Leer el archivo DBF con GeoPandas
dataframe = gpd.read_file(ruta_archivo)

# Extraer los valores de la columna "nationaCA"
valores_nationalCa = dataframe["nationalCa"].tolist()

# Imprimir los valores
for valor in valores_nationalCa:
    print(valor)

import os
import xml.etree.ElementTree as ET
from xml.dom import minidom
from datetime import datetime

# Assuming you already have the valores_nationalCa list

# Create the root element without attributes
root = ET.Element("LISTADATOS")
# Set the xmlns and xmlns:xsi attributes on the root element
root.set("xmlns", "http://www.catastro.meh.es/")
root.set("xmlns:xsi", "http://www.w3.org/2001/XMLSchema-instance")

# Get the current date
current_date = datetime.now().strftime("%d/%m/%Y")

# Add the FEC element with the current date
fec_element = ET.SubElement(root, "FEC")
fec_element.text = current_date

# Add the FIN element
fin_element = ET.SubElement(root, "FIN")
fin_element.text = "Consulta por Referencia Catastral"

# Add the DAT elements for each value in valores_nationalCa
for valor in valores_nationalCa:
    dat_element = ET.SubElement(root, "DAT")
    rc_element = ET.SubElement(dat_element, "RC")
    rc_element.text = valor

# Create the XML tree
xml_tree = ET.ElementTree(root)

# Define the directory where you want to save the XML file
output_directory =f"/content/drive/MyDrive/Catastro/{monte}/1.Archivos para consulta (Semilla)/Envio a ServMontes"
# Ensure the directory exists; create it if it doesn't
os.makedirs(output_directory, exist_ok=True)

# Specify the XML file name
xml_file_name = f"ConsultaMasiva_{monte}.xml"

# Construct the full file path
xml_file_path = os.path.join(output_directory, xml_file_name)

# Write the XML to the file
xml_tree.write(xml_file_path, encoding="utf-8", xml_declaration=True)

# Format the XML file to make it more readable (optional)
xml_dom = minidom.parse(xml_file_path)
formatted_xml = xml_dom.toprettyxml(indent="  ")

# Write the formatted XML back to the file
with open(xml_file_path, "w", encoding="utf-8") as f:
    f.write(formatted_xml)

# Print a message with the number of records written to the file
print(f"Numero de registros a consultar en el Servicio de Montes:{len(valores_nationalCa)}")

KeyError: 'nationalCa'


Una vez ya tenemos el archivo .xml generado lo comprimos en .zip para que se guarden en el mismo directorio de salida

In [ ]:
import zipfile
# Create a zip archive and add the formatted XML file to it
zip_file_name = f"ConsultaMasiva_{monte}.zip"
zip_file_path = os.path.join(output_directory, zip_file_name)

with zipfile.ZipFile(zip_file_path, "w") as zip_file:
    zip_file.write(xml_file_path, os.path.basename(xml_file_path))